In [ ]:
pip install qiskit qiskit-aer qiskit-ibm-runtime pylatexenc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.4/400.4 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.3/111.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.2/224.2 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 10.8 MB/s eta 0:00:00
  Created w

# Task 5.2: A Guide to Migrating from Qiskit Primitives V1 to V2

This notebook provides a practical, hands-on guide for updating your code from Qiskit's V1 Primitives to the modern V2 Primitives (`EstimatorV2`, `SamplerV2`).

**Note:** V1 Primitives are deprecated in recent Qiskit versions. This guide shows V1 code patterns as **commented examples** alongside **working V2 code** you can run locally with `AerSimulator`.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
import numpy as np

# The local simulator backend we will use for all examples
from qiskit_aer import AerSimulator

# --- V2 Primitives (Current/Recommended) ---
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler

# Tool to pretty-print options dictionaries
from dataclasses import asdict

# Define a local backend instance to use throughout
local_backend = AerSimulator()

print("Ready")


Ready


In [ ]:
# Helper function to create a Bell state circuit
def create_bell_circuit():
  """Create a Bell state (maximally entangled state) circuit."""
  qc = QuantumCircuit(2)
  qc.h(0)
  qc.cx(0, 1)
  return qc

# Test it
bell = create_bell_circuit()
print(bell)



     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘


## 1. The `run()` Method: Circuits, Observables, and Parameters

The most significant change from V1 to V2 is the structure of the `run()` method's inputs.

### Key Differences:
- **V1:** Took separate arguments for `circuits`, `observables`, and `parameter_values`
- **V2:** Takes a single list of **PUBs (Primitive Unified Blocs)** - tuples that group a circuit with its observables and parameters

### Estimator: V1 → V2 Migration

Let's see how to migrate a simple Bell circuit expectation value calculation.

In [ ]:
print("=" * 60)
print("V1 CODE (Deprecated - shown for reference):")
print("=" * 60)
print("""
# V1 Estimator - OLD WAY (Don't use this anymore)
from qiskit_ibm_runtime import Estimator

circuit = create_bell_circuit()
observable = SparsePauliOp("ZZ")

estimator = Estimator(backend=local_backend)
# V1 used separate keyword arguments
job = estimator.run(
    circuits=[circuit],
    observables=[observable]
)
result = job.result()
print(f"Expectation Value: {result.values[0]}")
""")

print("\n" + "=" * 60)
print("V2 CODE (Current - Working Example):")
print("=" * 60)

V1 CODE (Deprecated - shown for reference):

# V1 Estimator - OLD WAY (Don't use this anymore)
from qiskit_ibm_runtime import Estimator

circuit = create_bell_circuit()
observable = SparsePauliOp("ZZ")

estimator = Estimator(backend=local_backend)
# V1 used separate keyword arguments
job = estimator.run(
    circuits=[circuit],
    observables=[observable]
)
result = job.result()
print(f"Expectation Value: {result.values[0]}")


V2 CODE (Current - Working Example):


In [ ]:
# V2 Estimator - NEW WAY (Use this!)
from qiskit_ibm_runtime import EstimatorV2 as Estimator

circuit = create_bell_circuit()
observable = SparsePauliOp("ZZ")

estimator = Estimator(mode=local_backend)

# V2 uses PUBs: a list of tuples (circuit, observable, [optional params])
job = estimator.run(pubs=[(circuit, observable)])
result = job.result()

# V2 results are indexed by PUB, and data is in .data attribute
pub_result = result[0]
print(job.job_id())
print(pub_result.data.evs)
print(pub_result.data.stds)

a81f9d4b-514d-4c0a-a351-0667409897dd
1.0
0.0


### Sampler: V1 → V2 Migration

The pattern is the same for the Sampler.

In [ ]:
print("=" * 60)
print("V1 CODE (Deprecated - shown for reference):")
print("=" * 60)
print("""
# V1 Sampler - OLD WAY
from qiskit_ibm_runtime import Sampler

circuit = create_bell_circuit()
circuit.measure_all()

sampler = Sampler(backend=local_backend)
# V1 used separate keyword argument
job = sampler.run(circuits=[circuit])
result = job.result()
print(f"Quasi-distribution: {result.quasi_dists[0]}")
""")

print("\n" + "=" * 60)
print("V2 CODE (Current - Working Example):")
print("=" * 60)

V1 CODE (Deprecated - shown for reference):

# V1 Sampler - OLD WAY
from qiskit_ibm_runtime import Sampler

circuit = create_bell_circuit()
circuit.measure_all()

sampler = Sampler(backend=local_backend)
# V1 used separate keyword argument
job = sampler.run(circuits=[circuit])
result = job.result()
print(f"Quasi-distribution: {result.quasi_dists[0]}")


V2 CODE (Current - Working Example):


In [ ]:
# V2 Sampler - NEW WAY (Use this!)
from qiskit_ibm_runtime import SamplerV2 as Sampler

circuit = create_bell_circuit()
circuit.measure_all()

sampler = Sampler(mode=local_backend)

# V2 uses PUBs: a list of tuples (circuit, [optional params], [optional shots])
# Single circuit with no params - note the comma to make it a tuple

job = sampler.run(pubs=[(circuit)])
result = job.result()

# V2 results are indexed by PUB
pub_result = result[0]
print(job.job_id())
print(pub_result.data.meas.get_counts())
print(pub_result.data.meas.get_bitstrings()[:5])

63e0f405-1cb2-4bdf-b7e7-a3f94f0481dd
{'11': 520, '00': 504}
['11', '00', '00', '00', '11']


## 2. Handling Multiple Jobs with Different Parameters

The V2 PUB system makes it much easier to run varied jobs. You can specify different shots, parameters, or observables for each circuit.

### Example: Two Circuits with Different Shot Counts

In [ ]:
# Create two different circuits
circuit1 = create_bell_circuit()
circuit1.measure_all()

circuit2 = create_bell_circuit()
circuit2.ry(np.pi / 4, 0) # Add extra rotation
circuit2.measure_all()

# V2 PUB format for Sampler: (circuit, parameter_values, shots)
# We can specify different shots for each circuit!
pub1 = (circuit, None, 100)
pub2 = (circuit, None, 500)

sampler = Sampler(mode=local_backend)
job = sampler.run(pubs=[pub1, pub2])
result = job.result()


# Results are indexed corresponding to the input PUBs
print("circuit 1, shots : ", result[0].metadata['shots'])
print(result[0].data.meas.get_counts())
print("circuit 2")
print(result[1].data.meas.get_counts())

circuit 1, shots :  100
{'00': 56, '11': 44}
circuit 2
{'11': 249, '00': 251}


### Example: One Circuit with Multiple Observables

V2 makes it easy to measure multiple observables on the same circuit.

In [ ]:
circuit = create_bell_circuit()


# Define multiple observables to measure
obs1 = SparsePauliOp("ZZ")
obs2 = SparsePauliOp("XX")
obs3 = SparsePauliOp("YY")

# V2 PUB format: (circuit, [list of observables])
estimator = Estimator(mode=local_backend)
job = estimator.run(pubs=[(circuit, [obs1, obs2, obs3])])
result = job.result()

pub_result = result[0]
print("ZZ : ", pub_result.data.evs[0])
print("XX : ", pub_result.data.evs[1])
print("YY : ", pub_result.data.evs[2])

ZZ :  1.0
XX :  1.0
YY :  -1.0


## 3. Accessing Results (Output Structure Changes)

The result structure has changed significantly:

### V1 Results:
```python
# Estimator V1
result.values[0]           # Array of expectation values
result.metadata[0]         # Metadata for job 0

# Sampler V1  
result.quasi_dists[0]      # Quasi-probability distribution
```

### V2 Results:
```python
# Estimator V2
result[0].data.evs         # Expectation values for PUB 0
result[0].data.stds        # Standard deviations for PUB 0
result[0].metadata         # Metadata for PUB 0

# Sampler V2
result[0].data.meas.get_counts()      # Measurement counts
result[0].data.meas.get_bitstrings()  # Bitstring array
```

### Example: Converting V2 Sampler Output to V1-Style QuasiDistribution

In [ ]:
# Run a V2 Sampler job
circuit = create_bell_circuit()
circuit.measure_all()

sampler = Sampler(mode=local_backend)
job = sampler.run([(circuit,)], shots=1024)
result = job.result()

# V2 result structure
pub_result = result[0]
shots = pub_result.metadata['shots']
counts = pub_result.data.meas.get_counts()

print(counts)
print(shots)

v1_quasi_dist = {int(bitstring, 2): count/shots for bitstring, count in counts.items()}
print(f"\nV1-style QuasiDistribution (for migration): {v1_quasi_dist}")

{'00': 526, '11': 498}
1024

V1-style QuasiDistribution (for migration): {0: 0.513671875, 3: 0.486328125}


## 4. Setting Options

Options configuration has been streamlined in V2.

### V1 Approach (Deprecated):
```python
from qiskit_ibm_runtime import Options

options = Options()
options.resilience_level = 1
options.execution.shots = 4000
estimator = Estimator(backend=backend, options=options)

# Or update later:
estimator.set_options(resilience_level=2)
```

### V2 Approach (Current):
The primitive has a built-in `.options` attribute you can set directly.

In [ ]:
# Method 1: Pass options as a dictionary during initialization
estimator = Estimator(
    mode=local_backend,
    options={
        "resilience_level":1, "default_shots":2000
    }
)

print(estimator.options.resilience_level)
print(estimator.options.default_shots)

1
2000


In [ ]:
# Method 2: Set attributes directly (enables auto-complete in IDEs)
estimator = Estimator(mode=local_backend)
estimator.options.resilience_level = 1
estimator.options.default_shots = 4000

print(estimator.options.resilience_level)
print(estimator.options.default_shots)

1
4000


In [ ]:
# Method 3: Use update() for bulk updates
estimator = Estimator(mode=local_backend)
estimator.options.update(
    resilience_level=0,
    default_shots=1024
)

print(estimator.options.resilience_level)
print(estimator.options.default_shots)

print(asdict(estimator.options))

0
1024
{'_VERSION': 2, 'max_execution_time': Unset, 'environment': {'log_level': 'WARNING', 'job_tags': None, 'private': False}, 'simulator': {'noise_model': Unset, 'seed_simulator': Unset, 'coupling_map': Unset, 'basis_gates': Unset}, 'default_precision': Unset, 'default_shots': 1024, 'resilience_level': 0, 'seed_estimator': Unset, 'dynamical_decoupling': {'enable': Unset, 'sequence_type': Unset, 'extra_slack_distribution': Unset, 'scheduling_method': Unset, 'skip_reset_qubits': Unset}, 'resilience': {'measure_mitigation': Unset, 'measure_noise_learning': {'num_randomizations': Unset, 'shots_per_randomization': Unset}, 'zne_mitigation': Unset, 'zne': {'amplifier': Unset, 'noise_factors': Unset, 'extrapolator': Unset, 'extrapolated_noise_factors': Unset}, 'pec_mitigation': Unset, 'pec': {'max_overhead': Unset, 'noise_gain': Unset}, 'layer_noise_learning': {'max_layers_to_learn': Unset, 'shots_per_randomization': Unset, 'num_randomizations': Unset, 'layer_pair_depths': Unset}, 'layer_no

## 5. Summary: Migration Checklist

### Steps to Migrate to EstimatorV2:

1. **Update imports:**
   ```python
   # Old: from qiskit_ibm_runtime import Estimator
   # New:
   from qiskit_ibm_runtime import EstimatorV2 as Estimator
   ```

2. **Remove the Options import** (no longer needed)

3. **Change backend parameter:**
   ```python
   # Old: estimator = Estimator(backend=backend)
   # New:
   estimator = Estimator(mode=backend)
   ```

4. **Update options setting:**
   ```python
   # Old: estimator.set_options(resilience_level=1)
   # New:
   estimator.options.resilience_level = 1
   ```

5. **Group inputs into PUBs:**
   ```python
   # Old: job = estimator.run(circuits=[circuit], observables=[obs])
   # New:
   job = estimator.run(pubs=[(circuit, obs)])
   ```

6. **Update result access:**
   ```python
   # Old: result.values[0]
   # New:
   result[0].data.evs
   ```

### Steps to Migrate to SamplerV2:

1. **Update imports:**
   ```python
   from qiskit_ibm_runtime import SamplerV2 as Sampler
   ```

2. **Change backend parameter:**
   ```python
   sampler = Sampler(mode=backend)
   ```

3. **Update options:**
   ```python
   sampler.options.default_shots = 4096
   ```

4. **Group inputs into PUBs:**
   ```python
   # Old: job = sampler.run(circuits=[circuit])
   # New:
   job = sampler.run(pubs=[(circuit,)])  # Note the comma!
   ```

5. **Update result access:**
   ```python
   # Old: result.quasi_dists[0]
   # New:
   result[0].data.meas.get_counts()
   # or
   result[0].data.meas.get_bitstrings()
   ```